In [28]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

import os
from google.colab import drive

# 1. Bersihkan folder lokal 'drive' yang nyangkut (ghost folder)
if os.path.exists("/content/drive"):
    !rm -rf /content/drive

# 2. Lakukan mount Google Drive dengan force_remount
try:
    drive.mount("/content/drive", force_remount=True)
except Exception as e:
    print("Drive tidak terpasang:", e)

# 3. Buat direktori penyimpanan di Drive asli Anda
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum5"
os.makedirs(DIR_SIMPAN, exist_ok=True)

print(f"Direktori penyimpanan Praktikum 5 siap di: {DIR_SIMPAN}")

Mounted at /content/drive
Direktori penyimpanan Praktikum 5 siap di: /content/drive/MyDrive/BigData/Praktikum5


H-1. Peta Pipeline dan Permainan Kartu Kantin

H-2. Persiapan: Unduh Data dan Jalankan Portal Lokal

Dua sel berikut siap pakai. Sel pertama mengunduh CSV dan melihat ukurannya. Sel kedua menyalakan portal
berita mini di komputer kita. Kita tidak perlu memahami isi sel kedua, cukup tahu bahwa ia melayani halaman
/berita?hal=N berisi 100 judul per halaman, serta berkas robots.txt.

In [ ]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")

if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas:", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom:", df_sumber.shape)
print("Kolom :", list(df_sumber.columns))

import http.server, urllib.parse, html
HALAMAN_UKURAN = 100

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):
            pass # matikan log

        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>'
                    )
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    try:
        http.server.ThreadingHTTPServer.allow_reuse_address = True
        server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
        threading.Thread(target=server.serve_forever, daemon=True).start()
        return server, n_hal
    except OSError as e:
        if e.errno == 98:
            print(f"[INFO] Port {port} sudah digunakan. Menggunakan server yang sudah aktif.")
            return None, n_hal
        else:
            raise e

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Ukuran berkas: 16.1 MB
Baris, kolom: (91017, 4)
Kolom : ['date', 'url', 'title', 'category']
Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


H-3. Crawling Portal Lokal: Latihan Aman

---




Portal lokal selalu tersedia dan hasilnya sama untuk semua orang, sehingga cocok untuk berlatih mekanisme
crawling sebelum menyentuh situs sungguhan di H-4. Bila situs sungguhan nanti memblokir atau mengubah
susunannya, bagian ini tetap menjamin praktikum berjalan.


In [ ]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False
Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


H-4. Crawling Berita Sungguhan dari Sitemap detik.com


In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")

        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else "",
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else ""
        })
    return hasil

# Contoh XML untuk Uji
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9" xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
<url>
<loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
<news:news>
<news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
<news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
<news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
<news:keywords><![CDATA[mahkamah konstitusi, mk, hukum, kepala daerah]]></news:keywords>
</news:news>
</url>
<url>
<loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
<news:news>
<news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
<news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
</news:news>
</url>
<url>
<loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
<news:news>
<news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
<news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
</news:news>
</url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))

import urllib.robotparser

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: ddbinntang19@gmail.com)"
DELAY = 2.0
BATAS_PER_BAGIAN = 7
BATAS_TOTAL = 20

SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]
_robots = {}

def boleh_diambil(url, ua=UA_DETIK):
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")

    cek = pd.DataFrame(sampel)
    print("Judul terkumpul :", len(cek))
    print("Judul kosong    :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat    :", cek["url"].duplicated().sum())
    print("Tanggal kosong  :", cek["tanggal"].isna().sum())
    print("Kategori        :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi, mk, hukum, kepala daerah
Sama bila masukan berupa bytes: True
Judul terkumpul : 20
Judul kosong    : 0
URL duplikat    : 0
Tanggal kosong  : 0
Kategori        : {'news': 7, 'inet': 7, 'sport': 6}
                                                                     judul kategori    tanggal
      Pemerintah Sepakati Alokasi Pupuk Subsidi 2027 Sebesar 9,85 Juta Ton     news 2026-11-11
Zulhas Dorong Perlindungan 6,46 Juta Hektare Sawah untuk Swasembada Pangan     news 2026-11-11
        Ricuh 11.11 di

H-5. Channel ala Flume ke lapisan_mentah

Untuk skala penuh, seluruh 911 halaman portal lokal di-crawl (bukan detik.com, yang tidak boleh dibebani 91
ribu judul sekaligus). Crawler adalah source, antrean queue.Queue adalah channel (maksimal 2.000 event),
dan fungsi sink menulis setiap 5.000 event menjadi satu berkas JSONL di lapisan_mentah/. Source dan sink
berjalan di thread terpisah. Bila channel penuh, source otomatis menunggu.


In [ ]:
RAW = Path("lapisan_mentah"); RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0

    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []

    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0

print("Event masuk channel:", f"{statistik['masuk']:,}")
print("Event ditulis sink:", f"{statistik['keluar']:,}")
print("File batch        :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3], "...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel: 91,017
Event ditulis sink: 91,017
File batch        : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 45.3 detik untuk 911 halaman


H-6. Quality, Cleaning, dan Parquet Berpartisi

Seperti di Praktikum sebelumnya, kita mulai dengan memeriksa sebelum membersihkan. Fungsi
aturan_kualitas adalah bagian kunci: setiap aturan menghasilkan nilai benar/salah per baris.

In [ ]:
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))], ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"  : d["judul"].fillna("").str.strip() != "",
        "kategori dikenal"    : d["kategori"].isin(KATEGORI_DIKENAL),
        "judul 3-30 kata"     : jml_kata.between(3, 30),
        "tanggal valid"       : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid"   : d["url"].str.contains(r"/d-\d+", regex=True),
        "judul tidak duplikat": ~d["judul"].duplicated(keep=False)
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>6,} ({(~lulus).mean()*100:5.2f}%)")

# Proses pembersihan
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))

print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")

import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori finance")

Baris mentah: 91,017
judul tidak kosong       gagal:      0 ( 0.00%)
kategori dikenal         gagal:      0 ( 0.00%)
judul 3-30 kata          gagal:     78 ( 0.09%)
tanggal valid            gagal:      0 ( 0.00%)
url artikel valid        gagal:  5,640 ( 6.20%)
judul tidak duplikat     gagal:    235 ( 0.26%)
Baris setelah dibersihkan: 90,818 | dibuang: 199
partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori finance


H-7. Query DuckDB dan Bentuk Dokumen

Bacalah lapisan terkurasi dengan DuckDB seperti di Praktikum sebelumnya. Kolom kategori dibaca dari
nama folder (hive partitioning). Query kedua menghitung judul yang memuat kata "corona" per bulan.
Terakhir, satu baris diubah menjadi dokumen: bentuk fleksibel yang bisa menyimpan daftar kata di dalam
satu record.

In [ ]:
import duckdb

con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()
print(con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan
""").df().to_string(index=False))
print()

dokumen = {
    "id": int(bersih.iloc[0]["id"]) if pd.notna(bersih.iloc[0]["id"]) else None,
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


H-8. Mini MapReduce: Kasus Kantin

Sebelum memproses 90 ribu judul, kerjakan kasus kantin dalam kode. Tiga fungsi (map_fn, shuffle, reduce_fn)
mewakili tiga tahap, dan setiap tahap dicetak agar terlihat bagaimana data berubah bentuk.

In [ ]:
kasir = [
    ["nasi goreng, 1", "es teh, 2", "mie ayam, 1", "es teh, 1"],  # kasir A
    ["nasi goreng, 2", "es teh, 1", "es teh, 3", "mie ayam, 1"],  # kasir B
    ["mie ayam, 2", "nasi goreng, 1", "es teh, 1", "nasi goreng, 1"], # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu.strip(), int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# 4) COMBINER
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner:", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


H-9. Word Count Terdistribusi: Partisi sebagai Input Split


Sekarang pola yang sama diterapkan pada judul berita. Satu input split adalah satu berkas partisi Parquet,
jadi ada 9 mapper. Inilah jembatan dengan Praktikum sebelumnya, cara kita mempartisi data menentukan
cara pekerjaan dibagi. Perhatikan bahwa ukuran split sangat tidak seimbang (news memiliki 32 ribu judul,
sport hanya 2 ribu).

In [ ]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f" {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)

hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")
print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f" {k:<10} {n:>7,}")

top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f" {k:<10} {n:>7,}")

cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
 kategori=finance 14,141 judul
 kategori=food     4,314 judul
 kategori=health   4,914 judul
 kategori=hot     16,290 judul
 kategori=inet     5,577 judul
 kategori=news    32,321 judul
 kategori=oto      4,375 judul
 kategori=sport    2,433 judul
 kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
 di          27,390
 corona      17,239
 ini         10,124
 yang         8,668
 dan          6,785
 ke           5,592
 tak          5,096
 dari         4,697
 jadi         4,486
 untuk        3,723

10 kata teratas (tanpa stopword):
 corona      17,239
 rp           3,154
 virus        3,075
 kasus        2,743
 polisi       2,732
 positif      2,682
 indonesia    2,504
 warga        2,495
 rumah        2,465
 jokowi       2,317

Sama dengan Counter biasa: True


H-10. Combiner dan Jumlah Data yang Berpindah

Pertanyaan: apakah menjalankan mapper di beberapa proses membuat pekerjaan lebih cepat, dan apa
pengaruh combiner? Sel berikut mengukur waktu (median dari tiga kali jalan) dan menghitung pasangan
yang dikirim ke reducer. Angka pasangan stabil; waktu bervariasi antar-komputer dan antar-kali-jalan.

In [ ]:
from multiprocessing import Pool

def mapper_combiner(path):
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")

for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h):>20,}")

    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.553              825,601
1 proses, dengan combiner         0.697               74,009
2 proses, tanpa combiner          1.072              825,601
2 proses, dengan combiner         0.562               74,009

Hasil dengan combiner sama persis: True


H-11. Data Skew dan Beban Reducer

Partitioner memutuskan reducer mana yang menerima sebuah key. Kita memakai hash (CRC32) sehingga
hasilnya sama di setiap komputer. Sel berikut menghitung beban setiap reducer, lalu membandingkan reducer
terberat dengan bagian yang adil bila jumlah reducer ditambah.

In [ ]:
R = 4

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22} " + "  ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f" {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f" {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

tanpa combiner         R0=240,836 (29.2%)  R1=170,066 (20.6%)  R2=190,727 (23.1%)  R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%)  R1= 18,267 (24.7%)  R2= 18,530 (25.0%)  R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
   4 reducer: terberat 29.17% | bagian adil 25.00%
  16 reducer: terberat  8.82% | bagian adil  6.25%
  64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
 di        27,390 pasangan =  3.3% dari total, di reducer R0
 corona    17,239 pasangan =  2.1% dari total, di reducer R2
 ini       10,124 pasangan =  1.2% dari total, di reducer R0
 yang       8,668 pasangan =  1.0% dari total, di reducer R3
 dan        6,785 pasangan =  0.8% dari total, di reducer R3


H-13. Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman

Pada langkah terakhir, key diganti menjadi pasangan (kategori, kata), dengan stopword dibuang di sisi
mapper. Mapper memproses satu partisi, dan combiner dipasang di dalam mapper. Kemudian hasilnya
diekspor ke database SQLite, seperti sqoop export yang memindahkan hasil Hadoop ke database relasional.

In [ ]:
def mapper_kategori(path):
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))

for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

import sqlite3
SIAP = Path("siap_pakai"); SIAP.mkdir(exist_ok=True)

baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print("Baris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)
Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


Laihan


Stopword Domain beetujuanMenambahkan lima kata ke dalam stopword dan mengamati perubahan lima kata teratas pada setiap kategori.

In [ ]:
# Menyimpan hasil awal sebagai pembanding
top_awal = {}

for kat, data in per_kategori.items():
    top_awal[kat] = sorted(data, reverse=True)[:5]

# Menambahkan stopword baru
STOPWORD.update({"rp", "ri", "the", "foto", "galeri"})

# Menghitung ulang MapReduce per kategori
gabung_baru = collections.Counter()

for c in map(mapper_kategori, splits):
    gabung_baru.update(c)

# Menampilkan hasil setelah stopword diperbarui
per_kategori_baru = collections.defaultdict(list)

for (kat, kata), n in gabung_baru.items():
    per_kategori_baru[kat].append((n, kata))

for kat in sorted(per_kategori_baru):
    print(f"\nKategori: {kat}")

    print("Sebelum:")
    print(top_awal[kat])

    print("Sesudah:")
    print(sorted(
        per_kategori_baru[kat],
        reverse=True
    )[:5])


Kategori: finance
Sebelum:
[(2470, 'corona'), (1385, 'rp'), (1125, 'ri'), (701, 'harga'), (642, 'jokowi')]
Sesudah:
[(2470, 'corona'), (701, 'harga'), (642, 'jokowi'), (562, 'mau'), (536, 'ekonomi')]

Kategori: food
Sebelum:
[(699, 'makanan'), (539, 'makan'), (458, 'foto'), (438, 'galeri'), (358, 'enak')]
Sesudah:
[(699, 'makanan'), (539, 'makan'), (358, 'enak'), (343, 'bikin'), (311, 'rumah')]

Kategori: health
Sebelum:
[(2590, 'corona'), (1040, 'virus'), (471, 'pasien'), (427, 'indonesia'), (413, 'meninggal')]
Sesudah:
[(2590, 'corona'), (1040, 'virus'), (471, 'pasien'), (427, 'indonesia'), (413, 'meninggal')]

Kategori: hot
Sebelum:
[(1658, 'corona'), (793, 'anak'), (561, 'rumah'), (545, 'meninggal'), (528, 'the')]
Sesudah:
[(1658, 'corona'), (793, 'anak'), (561, 'rumah'), (545, 'meninggal'), (489, 'film')]

Kategori: inet
Sebelum:
[(845, 'corona'), (341, 'virus'), (309, 'ponsel'), (253, 'bikin'), (249, 'indonesia')]
Sesudah:
[(845, 'corona'), (341, 'virus'), (309, 'ponsel'), (253,

Analisis Frekuensi Kata per Bulan

Mengubah key MapReduce menjadi pasangan (bulan, kata) sehingga frekuensi kata dapat dianalisis berdasarkan bulan.

In [ ]:
hasil_bulan = collections.Counter()

for _, row in bersih.iterrows():
    bulan = row["bulan"]

    kata = tokenisasi(row["judul"])

    for k in kata:
        if k not in STOPWORD:
            hasil_bulan[(bulan, k)] += 1

# Menampilkan 10 kata teratas Januari dan Mei
for bulan in ["2020-01", "2020-05"]:
    print(f"\nKata teratas bulan {bulan}:")

    data = [
        (kata, jumlah)
        for (b, kata), jumlah in hasil_bulan.items()
        if b == bulan
    ]

    for kata, jumlah in sorted(
        data, key=lambda x: x[1], reverse=True
    )[:10]:
        print(kata, jumlah)


Kata teratas bulan 2020-01:
banjir 1220
polisi 554
jokowi 516
tahun 493
baru 475
mobil 428
minta 427
kpk 415
indonesia 410
china 386

Kata teratas bulan 2020-05:
corona 2935
kasus 640
positif 628
pandemi 596
warga 565
lebaran 562
covid 522
psbb 522
polisi 517
normal 504


Menghitung Bigram

Mengubah proses word count menjadi bigram, yaitu pasangan dua kata yang berurutan dalam satu judul.

In [ ]:
def tokenisasi_bigram(teks):
    kata = tokenisasi(teks)

    return [
        kata[i] + " " + kata[i + 1]
        for i in range(len(kata) - 1)
    ]

def mapper_bigram(path):
    judul = pd.read_parquet(
        path, columns=["judul"]
    )["judul"]

    c = collections.Counter()

    for t in judul:
        c.update(tokenisasi_bigram(t))

    return c

# Menghitung tanpa combiner
total_bigram = 0
hasil_bigram = collections.Counter()

for path in splits:
    judul = pd.read_parquet(
        path, columns=["judul"]
    )["judul"]

    for t in judul:
        total_bigram += len(tokenisasi_bigram(t))
        hasil_bigram.update(tokenisasi_bigram(t))

# Menghitung setelah combiner
hasil_combiner = [
    mapper_bigram(path) for path in splits
]

pasangan_combiner = sum(
    len(c) for c in hasil_combiner
)

total_hasil = collections.Counter()

for c in hasil_combiner:
    total_hasil.update(c)

print("Pasangan bigram tanpa combiner:",
      f"{total_bigram:,}")

print("Pasangan bigram setelah combiner:",
      f"{pasangan_combiner:,}")

print("Hasil tetap sama:",
      hasil_bigram == total_hasil)

print("\n10 bigram teratas:")

for kata, jumlah in hasil_bigram.most_common(10):
    print(kata, jumlah)

Pasangan bigram tanpa combiner: 734,783
Pasangan bigram setelah combiner: 411,217
Hasil tetap sama: True

10 bigram teratas:
virus corona 2915
corona di 2178
positif corona 1804
di tengah 1258
pandemi corona 1075
di rumah 1047
new normal 917
di indonesia 871
rp juta 700
hari ini 627


Perbandingan Jumlah Reducer

Mengetahui pengaruh jumlah reducer terhadap pembagian beban pemrosesan.

In [ ]:
print("Perbandingan beban reducer:")

for r in [2, 8, 32]:
    b = beban(hasil_map, r)

    total = sum(b)
    terberat = max(b)

    print(
        f"{r} reducer | "
        f"Terberat: {terberat / total * 100:.2f}% | "
        f"Ideal: {100 / r:.2f}%"
    )

Perbandingan beban reducer:
2 reducer | Terberat: 52.27% | Ideal: 50.00%
8 reducer | Terberat: 16.83% | Ideal: 12.50%
32 reducer | Terberat: 5.82% | Ideal: 3.12%


Crawling Ulang dengan Jeda

Mengukur waktu crawling ketika diberikan jeda sebesar 0,05 detik pada 50 halaman pertama.

In [ ]:
t0 = time.perf_counter()

hasil_crawl = list(
    crawl_iter(
        BASE,
        range(1, 51),
        delay=0.05
    )
)

waktu_50 = time.perf_counter() - t0

print("Jumlah berita:", len(hasil_crawl))
print("Waktu 50 halaman:",
      round(waktu_50, 2), "detik")

# Estimasi waktu untuk 911 halaman
overhead = max(
    waktu_50 - (50 * 0.05), 0
) / 50

estimasi = 911 * (1 + overhead)

print("Estimasi waktu 911 halaman dengan jeda 1 detik:",
      round(estimasi, 2), "detik")

Jumlah berita: 5000
Waktu 50 halaman: 4.65 detik
Estimasi waktu 911 halaman dengan jeda 1 detik: 950.2 detik


Menggunakan Sitemap dari Kategori Lain

Mengambil sitemap berita dari bagian lain dan memeriksa kualitas data menggunakan fungsi aturan_kualitas(). Pada bagian ini, alamat sitemap harus berasal dari daftar sitemap yang tersedia pada sitemap index detik.com. Jangan menggunakan URL yang dibuat sendiri tanpa memeriksa keberadaannya.

In [ ]:
# Masukkan URL sitemap XML yang valid, bukan URL halaman HTML biasa.
URL_SITEMAP_BARU = "https://news.detik.com/jabodetabek/sitemap_news.xml"

sesi = requests.Session()
sesi.headers["User-Agent"] = UA_DETIK

try:
    # Mengambil sitemap
    xml_baru = ambil_xml(URL_SITEMAP_BARU, sesi)

    # Parsing sitemap
    data_baru = parse_sitemap_news(xml_baru)[:20]

    # Cek apakah parsing berhasil (data tidak kosong)
    if not data_baru:
         print("Gagal: Sitemap tidak valid, kosong, atau XML tidak sesuai format.")
    else:
        df_baru = pd.DataFrame(data_baru)

        # Pemeriksaan kualitas
        print("Hasil Pemeriksaan Kualitas:")
        for nama, lulus in aturan_kualitas(df_baru).items():
            print(f"{nama:<24} | gagal: {(~lulus).sum()}")

        print("\nKategori yang ditemukan:")
        print(df_baru["kategori"].value_counts().to_string())

        print("\nContoh data:")
        print(df_baru[["judul", "kategori", "tanggal"]].head().to_string(index=False))

except Exception as e:
    print("Terjadi kesalahan saat mengambil sitemap:", type(e).__name__, "-", e)

Hasil Pemeriksaan Kualitas:
judul tidak kosong       | gagal: 0
kategori dikenal         | gagal: 11
judul 3-30 kata          | gagal: 0
tanggal valid            | gagal: 0
url artikel valid        | gagal: 11
judul tidak duplikat     | gagal: 0

Kategori yang ditemukan:
kategori
        11
news     9

Contoh data:
                                                                                    judul kategori    tanggal
Insiden Wadas, Dedi&amp;nbsp;Mulyadi&amp;nbsp;Harap Pemda Fasilitasi Mediasi Warga-Aparat     news 2022-02-10
                                Saat Buruh Tantang Ganjar: UMP Naik, Layak Jadi Presiden!     news 2021-11-27
                  Penampakan Mobil Ringsek Tertabrak Kereta di Pasuruan, Sekeluarga Tewas     news 2021-11-19
                       Geger Sosok Yana, Hilang di Cadas Pangeran Ditemukan di Majalengka     news 2021-11-19
                             Tolong Pak Jokowi, Ada TKI Sakit Minta Dipulangkan dari Irak     news 2021-09-27


Menghitung Kata Kunci Berita

Menghitung kata kunci yang paling sering muncul pada hasil crawling sitemap.

In [ ]:
def mapper_keyword(data):
    hasil = collections.Counter()

    for item in data:
        kata = item.get("kata_kunci", "")

        for k in kata.split(","):
            k = k.strip().lower()

            if k:
                hasil[k] += 1

    return hasil

def reducer_keyword(daftar_mapper):
    hasil = collections.Counter()

    for data in daftar_mapper:
        hasil.update(data)

    return hasil

# Mapper
hasil_mapper = mapper_keyword(sampel)

# Reducer
hasil_reducer = reducer_keyword([hasil_mapper])

print("10 kata kunci teratas:")

for kata, jumlah in hasil_reducer.most_common(10):
    print(kata, jumlah)

10 kata kunci teratas:
bnr 3
kemenko pangan 3
umkm 3
jensen huang 3
ibl all indonesian 2026 3
ibl all indonesian 3
bnr kemenko pangan 2
jabodetabek 2
perselingkuhan 2
ekonomi digital 2


Tugas Mandiri

Membuat mapper Inverted Index

In [ ]:
def mapper_inverted(data):
    hasil = []

    for index, row in data.iterrows():

        # Menggunakan ID asli jika tersedia
        if pd.notna(row["id"]):
            id_berita = str(int(row["id"]))

        # Membuat ID alternatif jika ID kosong
        else:
            id_berita = f"inet_{index:06d}"

        # Mengambil kata unik dari judul
        kata_unik = set(tokenisasi(row["judul"]))

        for kata in kata_unik:
            hasil.append((kata, id_berita))

    return hasil

Membuat proses Shuffle dan Reduce


In [ ]:
def reducer_inverted(pasangan):
    hasil = collections.defaultdict(set)

    for kata, id_berita in pasangan:
        hasil[kata].add(id_berita)

    return {
        kata: sorted(daftar_id)
        for kata, daftar_id in hasil.items()
    }

Menjalankan Inverted Index

In [ ]:
# Tahap Map
hasil_map = mapper_inverted(bersih)

print("Total pasangan hasil mapper:",
      f"{len(hasil_map):,}")

# Tahap Shuffle dan Reduce
inverted_index = reducer_inverted(hasil_map)

print("Jumlah kata unik:",
      f"{len(inverted_index):,}")

# Menampilkan tiga kata dengan jumlah dokumen terbanyak
top_kata = sorted(
    inverted_index.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:3]

print("\nTiga kata contoh dan lima ID pertama:")

for kata, daftar_id in top_kata:
    print(f"\nKata: {kata}")
    print("Jumlah dokumen:", len(daftar_id))
    print("Lima ID pertama:", daftar_id[:5])

Total pasangan hasil mapper: 817,993
Jumlah kata unik: 32,275

Tiga kata contoh dan lima ID pertama:

Kata: di
Jumlah dokumen: 26180
Lima ID pertama: ['4841806', '4841929', '4841931', '4841932', '4841936']

Kata: corona
Jumlah dokumen: 17228
Lima ID pertama: ['4856495', '4856598', '4868317', '4868545', '4869163']

Kata: ini
Jumlah dokumen: 10085
Lima ID pertama: ['4841859', '4841927', '4841929', '4841930', '4841936']


Menyimpan hasil Inverted Index

In [ ]:
with open(
    "siap_pakai/inverted_index.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        inverted_index,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Inverted Index berhasil disimpan.")

Inverted Index berhasil disimpan.


In [29]:
# Menyalin folder dan file dari penyimpanan sementara Colab ke Google Drive
!cp -r /content/contoh_crawl /content/drive/MyDrive/BigData/Praktikum5/
!cp -r /content/lapisan_mentah /content/drive/MyDrive/BigData/Praktikum5/
!cp -r /content/lapisan_terkurasi /content/drive/MyDrive/BigData/Praktikum5/
!cp -r /content/siap_pakai /content/drive/MyDrive/BigData/Praktikum5/
!cp /content/detik_news_title.csv /content/drive/MyDrive/BigData/Praktikum5/

print("Semua folder dan file praktikum berhasil disalin ke Drive!")

Semua folder dan file praktikum berhasil disalin ke Drive!
